# Video & Streaming

**TL;DR:** Expect "process this video": open it, read frames in a loop **with a
guard for failed reads**, do something per frame (motion, sample, annotate), and
**write an output video or frames**. The robust `cv2.VideoCapture` loop is the one
to have in muscle memory — it doubles as your fault-tolerance answer.

In [ ]:
%matplotlib inline
import cv2, numpy as np, matplotlib.pyplot as plt, os
from collections import deque

def show(img, title=""):
    """Display a BGR or grayscale image inline."""
    if img is None:
        print("image is None"); return
    plt.figure(figsize=(5,5))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

def make_sample_assets():
    """Create synthetic inputs so every cell runs without your own files.
    Swap input.jpg for a real photo any time to experiment."""
    doc = np.full((700,500,3), 30, np.uint8)
    quad = np.array([[120,90],[420,140],[400,560],[90,520]], np.int32)
    cv2.fillConvexPoly(doc, quad, (235,235,235))
    cv2.putText(doc, "LABEL 12345", (150,300), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20,20,20), 2)
    cv2.imwrite("input.jpg", doc)
    cv2.imwrite("noisy.jpg", doc)
    cv2.imwrite("low.jpg", (doc*0.3 + 90).astype(np.uint8))
    os.makedirs("ind", exist_ok=True)
    cv2.imwrite("ind/a.png", doc); cv2.imwrite("ind/b.png", doc)
    open("ind/c.png","wb").write(b"corrupt-not-an-image")
    vw = cv2.VideoWriter("test.avi", cv2.VideoWriter_fourcc(*"MJPG"), 10, (320,240))
    for i in range(12):
        fr = np.zeros((240,320,3), np.uint8)
        cv2.rectangle(fr, (10+i*15,80), (50+i*15,140), (255,255,255), -1)
        vw.write(fr)
    vw.release()
    return doc

doc = make_sample_assets()
print("Sample assets ready: input.jpg, noisy.jpg, low.jpg, ind/, test.avi")
show(doc, "synthetic input.jpg  (replace with your own image!)")


> **Tip:** run the setup cell above first. Each problem below defines its solution and then runs a quick demo. Edit and re-run any cell to experiment.

## Problem 13 — Robust capture: read, sample every Nth frame, save

**Tests:** the capture loop, failure handling, frame sampling, releasing resources.

In [ ]:
import cv2
import os

def sample_frames(source, out_dir="frames", every=15):
    """source: a video path or RTSP url. Saves every Nth frame as an image."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {source}")
    i = saved = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:                       # end of file OR dropped stream
                break
            if i % every == 0:
                cv2.imwrite(os.path.join(out_dir, f"frame_{i:05d}.jpg"), frame)
                saved += 1
            i += 1
    finally:
        cap.release()                        # ALWAYS release (no leaks)
    print(f"read {i} frames, saved {saved}")
    return saved

In [ ]:
print("saved", sample_frames("test.avi","frames",3), "frames"); show(cv2.imread("frames/frame_00000.jpg"), "a sampled frame")

**Watch out:** check `isOpened()`, handle `ok == False` (don't assume frames keep
coming), and `release()` in a `finally`. For live RTSP you'd wrap this in a
reconnect-with-backoff loop (see the fault-tolerance chapter).

---

## Problem 14 — Motion detection via frame differencing, write annotated video

**Tests:** per-frame processing, background diff, contours on a mask, `VideoWriter`.

In [ ]:
import cv2

def detect_motion(source, out="p14_out.mp4", min_area=800):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {source}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out, fourcc, fps, (w, h))
    prev = None
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY),
                                    (21, 21), 0)
            if prev is not None:
                diff = cv2.absdiff(prev, gray)
                th = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)[1]
                th = cv2.dilate(th, None, iterations=2)
                cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
                for c in cnts:
                    if cv2.contourArea(c) < min_area:
                        continue
                    x, y, bw, bh = cv2.boundingRect(c)
                    cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
            writer.write(frame)
            prev = gray
    finally:
        cap.release(); writer.release()
    return out

In [ ]:
detect_motion("test.avi","motion.avi"); print("wrote motion.avi")

**Watch out:** `VideoWriter` size must match the frames you write, or you get an
empty file. Blur before diff to kill sensor noise. `fourcc` "mp4v" for `.mp4`.

---

## Problem 15 — Overlay FPS / text and count frames

**Tests:** drawing text, reading video properties, simple per-frame overlay.

In [ ]:
import cv2, time

def annotate(source, out="p15_out.mp4"):
    cap = cv2.VideoCapture(source)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w, h = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    n, t0 = 0, time.time()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        n += 1
        real_fps = n / (time.time() - t0 + 1e-9)
        cv2.putText(frame, f"frame {n}  {real_fps:4.1f} fps", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        writer.write(frame)
    cap.release(); writer.release()
    return n

In [ ]:
print("frames annotated:", annotate("test.avi","annot.avi"))

**Watch out:** `putText` origin is the **bottom-left** of the text and coords are
(x, y). Colors are **BGR**.